# 1. データセット収集 (Pi0 / VLAモデル向け)
Genesisシミュレータを利用して、エキスパート（ヒューリスティックまたは手動操作）のデモンストレーションを収集し、学習用データセットを作成


In [1]:
#　torch　テスト
import torch
print(torch.cuda.is_available())

True


In [2]:
import genesis as gs
import numpy as np
import imageio
import torch
import os
from PIL import Image

# --- エラー回避用のモンキーパッチ (完全版) ---
# RTX 3070 Ti (Ampere) の仕様に合わせて不足しているプロパティを注入
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328

# 1. MuJoCo Menagerie（ロボット3Dモデル集）のダウンロード
if not os.path.exists("mujoco_menagerie"):
    !git clone https://github.com/google-deepmind/mujoco_menagerie.git

# 2. Genesisの初期化と環境構築
gs.init(backend=gs.cuda) 

scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

# 3. MJCF形式でUR5eを読み込み
robot = scene.add_entity(
    gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml")
)

cam = scene.add_camera(
    res=(640, 480),
    pos=(1.5, 1.5, 1.0),
    lookat=(0, 0, 0.5),
    fov=60,
    GUI=False
)

scene.build()
print('Genesis環境を構築しました。')

e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [19:45:47] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [19:45:47] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [19:45:47] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [19:45:47] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [19:45:47] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [19:45:47] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [19:45:47] [INFO] Scene <7ed124b> created.
[Genesis] [19:45:47] [INFO] Adding <gs.engine.entities.RigidEntity> plane_e46dafe. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [19:45:47] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [19:45:47] [WAR

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [19:47:04] [INFO] Building visualizer...
Genesis環境を構築しました。


In [8]:
def collect_data(num_episodes=2, steps_per_episode=100):
    os.makedirs('dataset', exist_ok=True)
    
    with h5py.File('dataset/demo_data.hdf5', 'w') as f:
        for ep in range(num_episodes):
            ep_group = f.create_group(f'episode_{ep}')
            obs_list, action_list = [], []
            
            # Genesis 1.4.x対応 API: set_dofs_position
            robot.set_dofs_position(np.zeros(robot.n_dofs))
            
            for step in range(steps_per_episode):
                action = np.random.uniform(-0.1, 0.1, size=robot.n_dofs)
                
                # Genesis 1.4.x対応 API: control_dofs_velocity
                robot.control_dofs_velocity(action)
                
                scene.step()
                
                # Genesis 1.4.x対応 API: get_dofs_position
                pos = robot.get_dofs_position()
                if torch.is_tensor(pos):
                    pos = pos.detach().cpu().numpy()
                    
                obs_list.append(pos)
                action_list.append(action)
                
            ep_group.create_dataset('observations', data=np.array(obs_list))
            ep_group.create_dataset('actions', data=np.array(action_list))
            
    print(f'データ収集完了: dataset/demo_data.hdf5 に {num_episodes} エピソード保存しました。')

# 実行
collect_data(num_episodes=2)

[Genesis] [19:50:40] [INFO] Running at 0.37 FPS.
[Genesis] [19:50:40] [INFO] Running at 8.31 FPS.
[Genesis] [19:50:40] [INFO] Running at 15.84 FPS.
[Genesis] [19:50:40] [INFO] Running at 21.92 FPS.
[Genesis] [19:50:40] [INFO] Running at 28.71 FPS.
[Genesis] [19:50:40] [INFO] Running at 37.12 FPS.
[Genesis] [19:50:40] [INFO] Running at 57.78 FPS.
[Genesis] [19:50:40] [INFO] Running at 74.72 FPS.
[Genesis] [19:50:40] [INFO] Running at 84.40 FPS.
[Genesis] [19:50:40] [INFO] Running at 107.39 FPS.
[Genesis] [19:50:40] [INFO] Running at 123.31 FPS.
[Genesis] [19:50:40] [INFO] Running at 135.93 FPS.
[Genesis] [19:50:41] [INFO] Running at 146.02 FPS.
データ収集完了: dataset/demo_data.hdf5 に 2 エピソード保存しました。
